# 07. Hands-on: Strands Decider — the Strands team's dedicated decision model

[strands-decider](https://github.com/strands-labs/strands-decider) is a **dedicated decision model** built by the Strands Agents team (Apache-2.0, HF `StrandsAgents` org — mirrored in the `amazon` org).

- Architecture: take a pretrained LLM torso (Qwen3.5-2B-Base or Gemma 4), **remove the LM head, add a ~1M-parameter pointer head**, adapt with LoRA — it answers in a single forward pass with no generation.
- Question types: noul (yes/no probability), choice (N-way), score (ordinal scale) + **calibrated confidence** (per the README: on unseen short classification tasks, answers at confidence >= 0.9 are right about 95% of the time).
- Server mode speaks **`POST /v1/systemone`** (the Jev/System One wire format) — the **same format** as the SGLang endpoint we have been using, so it plugs into our workflow with the same client.
- Official Strands Agents SDK integration: an `InterventionHandler` example asks the decider "are the arguments grounded / is this call premature?" right before each tool call and returns Proceed/Guide.

### Starting the server (in this workshop environment the GPU EC2 bootstrap already serves it on :8002)
```bash
pip install strands-decider            # a dedicated venv is recommended
strands-decider serve \
  StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8002 --device cuda
```


In [1]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import json, urllib.request
import ws_common as W

print(json.dumps(json.load(urllib.request.urlopen(f"http://{W.SERVER_HOST}:8002/health", timeout=5)), indent=1))


{
 "status": "ok",
 "model": "strands-decider-2B-qwen3.5-v1-2610",
 "checkpoint": "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610",
 "base_model": "Qwen/Qwen3.5-2B-Base",
 "num_slots": 24,
 "max_length": 4096,
 "temperature": 1.0545210242263665,
 "device": "cuda",
 "prefix_cache": true,
 "vision": false
}


## 1. Single decisions

Because the wire format is identical, you just point the **same `SGLangDecision` client at port 8002**. We ask D1/D2 with "My wallet was stolen. Please freeze my card."


In [2]:
from agent.decisions import SGLangDecision
from agent.prompts import D1_QUESTION, D2_QUESTION
from agent.state import CATEGORIES, D1_CHOICES

strands_dec = SGLangDecision(model="strands-decider", port=8002)   # reuse the same client
msg = "My wallet was stolen. Please freeze my card."
for q, ch, did in ((D1_QUESTION, D1_CHOICES, "d1"), (D2_QUESTION, CATEGORIES, "d2")):
    r = strands_dec.decide(state_text=msg, question=q, choices=ch, decision_id=did)
    print(f"[{did}] -> {r.label:<16s} confidence {r.confidence:.3f}   {r.latency_ms:,.1f}ms")

[d1] -> in_scope         confidence 0.600   89.2ms
[d2] -> card_security    confidence 0.699   80.5ms


**Observation**: D2 (routing) picks card_security with solid confidence (~0.70), while **D1 comes out noticeably less certain (~0.60)** on the 3-way safety call. That is the value of calibrated confidence — the model tells you in numbers how sure it is, so you can design a **cascade that escalates to a larger model below a confidence threshold**. (The Qwen3-VL you saw in 04 saturates at conf 1.0 even when it over-blocks, which makes that design impossible.)


## 2. Plugging it straight into our LangGraph workflow

Any decision backend with a matching `.decide()` signature is swappable. **Changing only the client's port — no code changes** — we run scenario 2 (delayed overseas transfer, the tool retry loop) on strands-decider decisions.

> Why scenario 2 instead of scenario 1 ("my card is not swiping"): strands-decider's D1 **mis-blocks** the scenario-1 question as out_of_scope (conf 0.498), ending the workflow after 2 STEPs. Try swapping it yourself — it is the lived experience of the D1 0.613 number in the quality table below.


In [3]:
from agent.graph_v2 import build_graph_v2

client, llm = W.make_clients()
graph_strands = build_graph_v2(strands_dec, llm)                     # decisions: strands-decider
graph_sglang  = build_graph_v2(SGLangDecision(port=8001), llm)       # decisions: SGLang Qwen3-VL (baseline)
sc = W.scenario(2)
print("Scenario:", sc["title"], "|", sc["question"])

Scenario: System error -> tool retry loop | The overseas transfer I sent yesterday afternoon has not reached the recipient yet. Reference number TR-20261026-8841, amount USD 2,000. Please check what happened.


In [4]:
res_strands = W.run_verbose(graph_strands, sc["question"], sc["script"], "StrandsDecider")


[StrandsDecider] customer: The overseas transfer I sent yesterday afternoon has not reached the recipient yet. Reference number TR-20261026-8841, amount USD 2,000. Please check what happened.

STEP 1 ▶ guardrail_check — D1 intake screening: in-scope / out-of-scope / compliance
      decision[D1] -> in_scope   (136ms, confidence 0.660)



STEP 2 ▶ intent_router — D2 routing: pick one of 12 specialist teams
      decision[D2] -> transfers   (134ms, confidence 0.943)

STEP 3 ▶ clarify_gate — D5 sufficiency check: can we act now, or must we ask back?
      decision[D5] -> info_sufficient   (135ms, confidence 0.539)



STEP 4 ▶ tool_step — D3 tool choice -> run tool -> D6 result review (done/retry/give up)
      decision[D3] -> track_transfer   (86ms, confidence 0.935)
      tool run: track_transfer   (0.0ms)
      decision[D6] -> try_alternative   (144ms, confidence 0.734)



STEP 5 ▶ tool_step — D3 tool choice -> run tool -> D6 result review (done/retry/give up)
      decision[D3] -> transfer_info   (87ms, confidence 0.876)
      tool run: transfer_info   (0.0ms)
      decision[D6] -> goal_met   (891ms, confidence 0.721)



STEP 6 ▶ summarize — Sonnet condenses the tool result into customer-facing facts
      generate (Sonnet 4.5)   (5,630ms)
      summary (first line): Based on the tool result provided, I cannot extract specific resolution facts ab

STEP 7 ▶ risk_check — D7 risk check: fraud/AML monitoring flag present?
      decision[D7] -> no_risk   (150ms, confidence 0.759)



STEP 8 ▶ escalation_gate — D4 final action: resolve / clarify / human handoff
      decision[D4] -> clarify   (157ms, confidence 0.095)



STEP 9 ▶ respond — Sonnet writes the final customer reply
      generate (Sonnet 4.5)   (4,687ms)
      --- final reply ---
      Thank you for contacting us about your international transfer.
      
      To investigate the status of your transfer (Reference: TR-20261026-8841, USD 2,000), I need a bit more information:
      
      1. **Recipient's bank name and country** - Where was the money sent?
      2. **Recipient's account details** - Can you confirm the beneficiary name and account number you used?
      3. **Transfer method** - Was this a SWIFT transfer or another service?
      
      International transfers typically take 1-5 business days depending on the destination country and correspondent banks involved. Since you sent it yesterday afternoon, it may still be in transit.
      
      Once you provide these details, I can check the exact status and processing timeline for your transfer.

>>> [StrandsDecider] 9 decisions, total 1,920ms | generation 10,317ms | e2e 12,247m

In [5]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")


[SGLang] customer: The overseas transfer I sent yesterday afternoon has not reached the recipient yet. Reference number TR-20261026-8841, amount USD 2,000. Please check what happened.

STEP 1 ▶ guardrail_check — D1 intake screening: in-scope / out-of-scope / compliance
      decision[D1] -> in_scope   (37ms, confidence 1.000)

STEP 2 ▶ intent_router — D2 routing: pick one of 12 specialist teams
      decision[D2] -> transfers   (34ms, confidence 1.000)

STEP 3 ▶ clarify_gate — D5 sufficiency check: can we act now, or must we ask back?
      decision[D5] -> info_sufficient   (33ms, confidence 1.000)

STEP 4 ▶ tool_step — D3 tool choice -> run tool -> D6 result review (done/retry/give up)
      decision[D3] -> track_transfer   (32ms, confidence 1.000)
      tool run: track_transfer   (0.0ms)
      decision[D6] -> try_alternative   (35ms, confidence 1.000)



STEP 5 ▶ tool_step — D3 tool choice -> run tool -> D6 result review (done/retry/give up)
      decision[D3] -> transfer_info   (33ms, confidence 1.000)
      tool run: transfer_info   (0.0ms)
      decision[D6] -> goal_met   (34ms, confidence 0.665)



STEP 6 ▶ summarize — Sonnet condenses the tool result into customer-facing facts
      generate (Sonnet 4.5)   (4,412ms)
      summary (first line): Based on the tool result provided, I cannot extract specific resolution facts ab

STEP 7 ▶ risk_check — D7 risk check: fraud/AML monitoring flag present?
      decision[D7] -> no_risk   (38ms, confidence 1.000)

STEP 8 ▶ escalation_gate — D4 final action: resolve / clarify / human handoff
      decision[D4] -> human_handoff   (37ms, confidence 0.726)



STEP 9 ▶ respond — Sonnet writes the final customer reply
      generate (Sonnet 4.5)   (4,343ms)
      --- final reply ---
      Thank you for providing your transfer details. I can see reference number TR-20261026-8841 for USD 2,000, but I need to escalate this to our specialized international payments team to fully investigate the delay.
      
      A specialist will contact you within 2-4 business hours via your registered email or phone number. They will have access to complete tracking information and can provide you with specific details about your transfer's current status and expected arrival time.
      
      Please keep your reference number TR-20261026-8841 handy for when they reach out.
      
      We apologize for any inconvenience this delay may have caused.

>>> [SGLang] 9 decisions, total 312ms | generation 8,755ms | e2e 9,075ms


In [6]:
W.compare(res_strands, res_sglang)

| metric | StrandsDecider | SGLang | ratio(StrandsDecider/SGLang) |
|---|---:|---:|---:|
| decisions | 9 | 9 | |
| **decision latency total** | **1,920ms** | **312ms** | **6.1x** |
| generation (LLM) latency | 10,317ms | 8,755ms | **1.2x** |
| e2e | 12,247ms | 9,075ms | **1.3x** |

**decision-layer speedup: 6.1x** (e2e saved 3,172ms)

**per-decision latency**

| node | decision | StrandsDecider | SGLang | ratio | label |
|---|---|---:|---:|---:|---|
| guardrail_check | D1 | 136ms | 37ms | **3.7x** | in_scope (same) |
| intent_router | D2 | 134ms | 34ms | **3.9x** | transfers (same) |
| clarify_gate | D5 | 135ms | 33ms | **4.1x** | info_sufficient (same) |
| tool_select | D3 | 86ms | 32ms | **2.7x** | track_transfer (same) |
| tool_review | D6 | 144ms | 35ms | **4.1x** | try_alternative (same) |
| tool_select | D3 | 87ms | 33ms | **2.7x** | transfer_info (same) |
| tool_review | D6 | 891ms | 34ms | **26.0x** | goal_met (same) |
| risk_check | D7 | 150ms | 38ms | **4.0x** | no_risk (same) |
| escalation_gate | D4 | 157ms | 37ms | **4.3x** | StrandsDecider=clarify / SGLang=human_handoff |

## Measured decision quality on the Korean bench (3-backend comparison)

D1/D2 measured over the workshop's 160-conversation Korean bench (the [KR] track inputs) with objective gold labels (the source project's decider-quality bench, the bench summary (source project)):

| backend | D1 guardrail (3-way) | D2 routing strict/lenient (12-way) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**Reading**: model capacity dominated the quality ceiling (8B > 2B > 0.3B), beating language specialization (Laya multilingual). The Laya number matches its own documented zero-shot score (0.362); Laya is a **"fine-tune me" model** — RLCD fine-tuning lifts it to 0.766. Conversely, smaller means better latency and footprint — a clean three-way quality/cost/speed trade-off.


## 3. Wrap-up — where is this the right choice?

| Angle | strands-decider (dedicated 2B) | SGLang + Qwen3-VL (endpoint added to a served LLM) |
|---|---|---|
| Deployment | One small dedicated model (CPU-capable, ~489ms/question per README) | Reuses the LLM/VLM you already serve (~zero marginal cost) |
| Confidence | **Calibrated** (low confidence = genuinely unsure) -> easy cascade design | Tends to saturate (hard to design thresholds) |
| Multimodal | Text only (vision=false) | Image decisions possible (lab 05) |
| Ecosystem | Official Strands Agents SDK intervention example | Broad System One wire-format compatibility |

- Decision latency in this environment (GPU shared with SGLang) is **~80ms** warm — slower than SGLang (~35ms) but far faster than Bedrock LLM decisions (~640ms).
- On the Korean bench the 2B torso is "usable but unsure of itself" — and the confidence says so honestly, which makes a **low threshold + larger-model fallback** the natural setup for Korean decisions.
